In [3]:
import pandas as pd
import matplotlib as plt

from Project.Choropleth_crime import amsterdam_crime

df = pd.read_csv("data/total_table.csv")

df


,wijkcode,wijknaam,aantal_lantaarns,Totaal misdrijven,wijkcode_clean,AfstandTotHuisartsenpraktijk_5,AfstandTotHuisartsenpost_9,AfstandTotApotheek_10,AfstandTotZiekenhuis_11,AfstandTotZiekenhuis_15,...,AfstandTotZwembad_93,AfstandTotKunstijsbaan_94,AfstandTotMuseum_95,AfstandTotPodiumkunstenTotaal_99,AfstandTotPoppodium_103,AfstandTotBioscoop_104,AfstandTotSauna_108,AfstandTotZonnebank_109,AfstandTotAttractie_110,AfstandTotBrandweerkazerne_114
0,WK0363AA,Haarlemmerbuurt,885,73.0,wk0363aa,0.5,4.5,0.7,2.0,4.5,...,1.4,6.5,1.2,0.7,2.6,0.9,1.5,0.8,3.3,1.9
1,WK0363AB,Jordaan,1330,150.0,wk0363ab,0.4,3.7,0.5,1.5,3.7,...,0.8,6.8,0.6,0.6,1.5,1.0,1.1,0.8,3.2,0.9
2,WK0363AC,Grachtengordel-West,665,92.0,wk0363ac,0.4,3.9,0.7,1.0,3.9,...,1.2,6.3,0.5,0.5,1.4,1.2,0.6,0.8,3.1,1.3
3,WK0363AD,Burgwallen-Nieuwe Zijde,902,452.0,wk0363ad,0.5,3.7,0.7,0.8,3.7,...,1.6,6.0,0.5,0.8,1.7,1.4,0.5,1.2,2.7,1.7
4,WK0363AE,Burgwallen-Oude Zijde,861,259.0,wk0363ae,0.5,3.2,0.4,1.1,3.2,...,2.1,5.4,0.3,0.5,2.1,1.2,0.5,1.7,2.2,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,WK0363TH,K-buurt,1308,29.0,wk0363th,0.5,4.0,0.8,4.0,4.0,...,2.3,6.3,5.8,2.2,7.6,3.7,4.2,4.3,3.3,1.8
106,WK0363TJ,Holendrecht,1905,40.0,wk0363tj,0.9,1.4,0.9,1.4,1.4,...,2.6,8.1,6.6,2.5,9.1,3.5,6.1,6.4,2.9,2.4
107,WK0363TK,Nellestein,1059,18.0,wk0363tk,0.9,3.5,1.5,3.5,3.5,...,2.7,7.1,5.9,2.6,8.4,4.1,5.0,5.3,2.6,0.6
108,WK0363TL,Reigersbos,1743,37.0,wk0363tl,1.0,2.5,1.0,2.5,2.5,...,3.7,8.7,7.5,3.6,9.8,4.6,6.8,7.1,2.6,2.3


In [6]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


amsterdam_crime = pd.read_csv("data/df_crime_average_per_year.csv")
# 1. Zorg dat alles in het juiste CRS staat (RD New)
# (Ik ga ervan uit dat je dataframes 'wijken', 'bomen', 'lichtpunten', 'amsterdam_crime' al geladen zijn zoals eerder)

# Bereken oppervlakte in km2 (area geeft m2 in RD stelsel)
amsterdam_crime['oppervlakte_km2'] = amsterdam_crime.geometry.area / 1_000_000

# 2. Tel Bomen per Wijk (Spatial Join had je al gedaan, nu tellen)
# We doen een nieuwe join voor de zekerheid om de telling te krijgen
bomen_in_wijken = gpd.sjoin(bomen, amsterdam_crime[['wijknaam', 'geometry']], how='inner', predicate='within')
boom_telling = bomen_in_wijken.groupby('wijknaam').size().reset_index(name='aantal_bomen')

# 3. Tel Lichtpunten per Wijk
licht_in_wijken = gpd.sjoin(lichtpunten, amsterdam_crime[['wijknaam', 'geometry']], how='inner', predicate='within')
licht_telling = licht_in_wijken.groupby('wijknaam').size().reset_index(name='aantal_lichtpunten')

# 4. Alles samenvoegen in één dataframe voor analyse
analyse_df = amsterdam_crime.merge(boom_telling, on='wijknaam', how='left')
analyse_df = analyse_df.merge(licht_telling, on='wijknaam', how='left')

# Vul NaNs met 0
analyse_df[['aantal_bomen', 'aantal_lichtpunten']] = analyse_df[['aantal_bomen', 'aantal_lichtpunten']].fillna(0)

# 5. BEREKEN DICHTHEDEN (Normalisatie)
analyse_df['boom_dichtheid'] = analyse_df['aantal_bomen'] / analyse_df['oppervlakte_km2']
analyse_df['licht_dichtheid'] = analyse_df['aantal_lichtpunten'] / analyse_df['oppervlakte_km2']
analyse_df['misdaad_dichtheid'] = analyse_df['Totaal misdrijven'] / analyse_df['oppervlakte_km2']

print(analyse_df[['wijknaam', 'misdaad_dichtheid', 'boom_dichtheid', 'licht_dichtheid']].head())

AttributeError: 'DataFrame' object has no attribute 'geometry'

In [4]:
# Selecteer de interessante kolommen (Afstand tot...)
distance_cols = [col for col in df.columns if 'Afstand' in col]

# We willen correleren met de MISDAADDICHTHEID (misdaad per km2), dat is eerlijker dan totaal aantal.
# (Zorg dat 'misdaad_dichtheid' berekend is zoals in het vorige antwoord)
correlations = df[distance_cols + ['misdaad_dichtheid']].corr()['misdaad_dichtheid'].drop('misdaad_dichtheid')

# Sorteer van sterkste negatieve correlatie (dichtbij = veel misdaad) naar positief
correlations = correlations.sort_values()

# Plotten
plt.figure(figsize=(10, 12))
correlations.plot(kind='barh', color='teal')
plt.title("Correlatie: Afstand tot Voorziening vs. Misdaaddichtheid")
plt.xlabel("Correlatiecoëfficiënt (Links = Dichterbij zorgt voor meer misdaad)")
plt.axvline(x=0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

KeyError: "['misdaad_dichtheid'] not in index"